# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described using a Croissant schema at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"License: {metadata.license}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")


## 2. Data Overview
Review available record sets, their fields (columns), and their `@id`s. All identifiers are referenced by `@id` for clarity and reproducibility.

In [ ]:
# List record sets and their fields using their @id.
record_sets = []
for rs in dataset.metadata.recordSets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, type: {getattr(field, 'dataType', 'unknown')})")
    print()
    record_sets.append(rs.id)
if not record_sets:
    print("No record sets defined in the Croissant metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All lookups are performed by the `@id` of the record set.

In [ ]:
# Select one or more available record set IDs for extraction (from the lists above).
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"Loading data for RecordSet: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records with columns:", df.columns.tolist())
            display(df.head())
        except Exception as e:
            print(f"Could not load record set {record_set_id}: {e}")
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Analyze numeric fields in the dataset, filter records by values, normalize numeric columns, and group data. All operations reference fields and record sets by their `@id`.

For this demonstration, we select the main record set (if available) and a numeric field (e.g., protein molecular weight or abundance), referenced by `@id`.

In [ ]:
from IPython.display import display

# Select the main record set if available
if record_sets:
    selected_record_set_id = record_sets[0]  # You can change the index to explore others
    df = dataframes[selected_record_set_id]
    print(f"Selected record set: {selected_record_set_id}")
    # List numeric candidate columns by @id or name
    print("\nColumns in this record set:", df.columns.tolist())
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if not numeric_candidates:
        print("No numeric columns found for EDA.")
    else:
        # Pick the first numeric column as an example
        numeric_field_id = numeric_candidates[0]
        print(f"\nUsing numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0

        # Filtering
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nRecords where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized column '{norm_col}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping (by the first non-numeric field, if any)
        group_fields = df.select_dtypes(exclude=['float', 'int']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by field (by @id): {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped.head())
        else:
            print("\nNo non-numeric field to group by.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships. For demonstration, we'll plot a histogram of the selected numeric field and a boxplot by group (if applicable).

In [ ]:
# Visualization using matplotlib or seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if available)
    if group_fields:
        group_field_id = group_fields[0]
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No suitable numeric field for plotting.")

## 6. Conclusion
- The dataset was described and explored using the `mlcroissant` library.
- Record sets and fields were accessed by their `@id`.
- We loaded records into DataFrames, filtered and normalized numeric data, and visualized field distributions.
- For deeper analysis, inspect specific `@id`s, fields, or use full protein descriptors present in the dataset.